In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# Some parameters for MHA
embed_dimension = 10
n_samples = 1
n_particles = 3
num_heads = 1

In [3]:
# Some dummy inputs
x = torch.rand(n_particles, n_samples, embed_dimension)
print(x)
print(x.shape)
attention_mask = torch.tensor([[[0,1,1], [1,0,1], [1,1,0]]], dtype=torch.float)
key_mask = torch.tensor([[0,1,0]], dtype=torch.bool)
print(attention_mask)
print(key_mask)

tensor([[[0.7803, 0.7348, 0.3172, 0.2579, 0.5393, 0.8287, 0.2745, 0.3423,
          0.3423, 0.9051]],

        [[0.0743, 0.8450, 0.0679, 0.0543, 0.0771, 0.9784, 0.8512, 0.7689,
          0.6135, 0.5514]],

        [[0.6671, 0.9668, 0.3148, 0.9129, 0.2587, 0.6223, 0.1094, 0.5242,
          0.6873, 0.5702]]])
torch.Size([3, 1, 10])
tensor([[[0., 1., 1.],
         [1., 0., 1.],
         [1., 1., 0.]]])
tensor([[False,  True, False]])


In [4]:
# Initialize a MHA layer
mha = nn.MultiheadAttention(embed_dimension, num_heads=num_heads, bias=False)

In [5]:
# Forward pass of MHA
y, weights = mha.forward(x, x, x, average_attn_weights=False)
print(y)
print(y.shape)
print(weights)

tensor([[[-0.1182, -0.1304, -0.0990, -0.3383, -0.1591, -0.4186, -0.4795,
           0.0394, -0.2005, -0.0334]],

        [[-0.1161, -0.1320, -0.0974, -0.3407, -0.1570, -0.4202, -0.4832,
           0.0368, -0.2030, -0.0330]],

        [[-0.1168, -0.1314, -0.0998, -0.3397, -0.1594, -0.4203, -0.4815,
           0.0397, -0.2007, -0.0326]]], grad_fn=<ViewBackward0>)
torch.Size([3, 1, 10])
tensor([[[[0.3254, 0.3327, 0.3419],
          [0.3258, 0.3179, 0.3563],
          [0.3186, 0.3294, 0.3519]]]], grad_fn=<ViewBackward0>)


In [210]:
x_new = x
# x_new[0,0,:] = torch.zeros_like(x_new[0,0,:])
# print(x_new)
y_mod, weights = mha.forward(x_new, x_new, x_new, average_attn_weights=False, key_padding_mask=key_mask, attn_mask=attention_mask)
print(y_mod)
print(y_mod.shape)
print(weights)

tensor([[[-0.4250,  0.0398, -0.3048,  0.2119, -0.0194, -0.0331, -0.0089,
           0.1299,  0.0761,  0.2241]],

        [[-0.4236,  0.0664, -0.3401,  0.1997, -0.0148, -0.0499,  0.0125,
           0.1402,  0.0902,  0.2307]],

        [[-0.4264,  0.0141, -0.2706,  0.2237, -0.0238, -0.0169, -0.0296,
           0.1199,  0.0625,  0.2176]]], grad_fn=<ViewBackward0>)
torch.Size([3, 1, 10])
tensor([[[[0.0000, 0.4950, 0.5050],
          [0.0000, 0.2590, 0.7410],
          [0.0000, 0.7234, 0.2766]]]], grad_fn=<ViewBackward0>)


/Users/kevingreif/anaconda3/envs/zjets_ur/lib/python3.11/site-packages/torch/nn/functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [10]:
# Implement using my own attention using the initialized weights
ipw = mha.in_proj_weight
q_w = x[0:1,0,:] @ ipw[:10,:].t()
k_w = x[:,0,:] @ ipw[10:20,:].t()
v_w = x[:,0,:] @ ipw[20:30,:].t()
print(q_w.shape)
print(k_w.shape)
print(v_w.shape)
softmax_output = torch.softmax((q_w @ k_w.t()) / 3, dim=1)
print(softmax_output)
attention = softmax_output @ v_w
# print(attention)
final_output = attention @ mha.out_proj.weight.t()
print(final_output)
print(final_output.shape)

torch.Size([1, 10])
torch.Size([3, 10])
torch.Size([3, 10])
tensor([[0.3250, 0.3327, 0.3424]], grad_fn=<SoftmaxBackward0>)
tensor([[-0.1182, -0.1304, -0.0991, -0.3384, -0.1591, -0.4187, -0.4796,  0.0394,
         -0.2005, -0.0333]], grad_fn=<MmBackward0>)
torch.Size([1, 10])
